In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: source_0
position:
  x: 190
  y: -84.5
description:
  text: Load data from a specified file or table, optionally selecting dimensions and measures.
  hash: b215ea46
previewMode: "1000"
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: filter_1
template: filter
templateVersion: 2.0.0
name: filter_1
position:
  x: 424
  y: 191
description:
  text: Keep all rows since no condition is set.
  hash: 6bc246d6
previewMode: "1000"
config:
  condition: ""
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": ""
}
inputs = {
    "data": ctx["source_0.data"]
}
out = run(config, inputs, spark)
ctx["filter_1.filtered_data"] = out["filtered_data"]
ctx["filter_1.excluded_data"] = out["excluded_data"]

In [0]:
"""
id: prepare_2
template: prepare
templateVersion: 1.0.0
name: prepare_2
position:
  x: 678
  y: -125
previewMode: "1000"
config:
  actions: []
input:
  - node: filter_1
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "actions": []
}
inputs = {
    "data": ctx["filter_1.filtered_data"]
}
out = run(config, inputs, spark)
ctx["prepare_2.prepared_data"] = out["prepared_data"]

In [0]:
"""
id: sql_3
template: sql
templateVersion: 1.0.0
name: sql_3
position:
  x: 915
  y: -180.666015625
description:
  text: Keep all data from prepare_2.
  hash: a5a0ced0
previewMode: "1000"
config:
  query: SELECT * FROM prepare_2
input:
  - node: prepare_2
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT * FROM prepare_2"
}
inputs = {
    "data": [
        ctx["prepare_2.prepared_data"]
    ],
    "data__sources": [
        {
            "node": "prepare_2",
            "output_port": "prepared_data",
            "name": "prepare_2",
            "df_name": "prepare_2"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_3.result"] = out["result"]

In [0]:
"""
id: hive_metastore.default.output_4
template: output
templateVersion: 1.0.0
name: hive_metastore.default.output_4
position:
  x: 1287
  y: -179.666015625
description:
  text: Save data by replacing existing table in the specified location.
  hash: f9040daa
previewMode: "1000"
config:
  catalog: hive_metastore
  schema: "`default`"
  table_name: output_4
input:
  - node: sql_3
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "hive_metastore",
    "schema": "`default`",
    "table_name": "output_4"
}
inputs = {
    "data": ctx["sql_3.result"]
}
out = run(config, inputs, spark)